<a href="https://colab.research.google.com/github/sisi-y/100-Days-ML-Journey/blob/main/KenyaLoanDefault.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install xgboost -q
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
print(f"XGBoost version: {xgb.__version__}")
print("✅ All imports successful")

XGBoost version: 3.2.0
✅ All imports successful


In [11]:
np.random.seed(42)
n = 2000
data = {
    'age': np.random.randint(22, 65, n),
    'monthly_income_kes': np.random.randint(15000, 250000, n),
    'loan_amount_kes': np.random.randint(10000, 500000, n),
    'employment_years': np.random.randint(0, 30, n),
    'credit_score': np.random.randint(300, 850, n),
    'num_dependants': np.random.randint(0, 8, n),
    'has_savings_account': np.random.randint(0, 2, n),
    'county': np.random.choice(
        ['Nairobi', 'Mombasa', 'Kisumu', 'Nakuru', 'Eldoret'], n
    )
}
df = pd.DataFrame(data)
risk_score =(
  (df['loan_amount_kes'] / df['monthly_income_kes']) * 0.4 +
    (850 - df['credit_score']) / 850 * 0.3 +
    df['num_dependants'] / 8 * 0.2 -
    df['employment_years'] / 30 * 0.1
)
df['defaulted'] = (risk_score + np.random.normal(0, 0.1, n) > 0.5).astype(int)

print(df.head())
print(f"\nShape: {df.shape}")
print(f"Default rate: {df['defaulted'].mean():.1%}")

   age  monthly_income_kes  loan_amount_kes  employment_years  credit_score  \
0   60              227781           241324                16           769   
1   50              205961           334512                 5           805   
2   36              173059           102122                18           492   
3   64               93714            54760                23           367   
4   29              204420           295622                 5           793   

   num_dependants  has_savings_account   county  defaulted  
0               5                    1   Nakuru          1  
1               0                    0  Mombasa          1  
2               7                    1   Kisumu          0  
3               0                    1  Eldoret          0  
4               4                    0  Eldoret          1  

Shape: (2000, 9)
Default rate: 79.7%


In [12]:
le = LabelEncoder()
df['county_encoded'] = le.fit_transform(df['county'])
feature_cols = ['age', 'monthly_income_kes', 'loan_amount_kes', 'employment_years', 'credit_score', 'num_dependants',
                'has_savings_account', 'county_encoded']
X= df[feature_cols]
y = df['defaulted']

X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.2, random_state=42,stratify=y)
print(f"Training rows: {X_train.shape[0]}")
print(f"Test rows:     {X_test.shape[0]}")
print("✅ Note: No scaling needed — XGBoost is tree-based")

Training rows: 1600
Test rows:     400
✅ Note: No scaling needed — XGBoost is tree-based


In [13]:
xgb_es = XGBClassifier(
   n_estimators=1000,
   learning_rate=0.05,
   max_depth=5,
   subsample= 0.8,
   colsample_bytree=0.8,
   reg_alpha=0.1,
   reg_lambda=1.0,
   random_state=42,
   eval_metric='logloss',
   verbosity=0
)

# The early_stopping_callback definition remains but will not be used in the fit method.
# early_stopping_callback = xgb.callback.EarlyStopping(
#     rounds=50,
#     save_best=True,
#     maximize=False, # For logloss, we want to minimize it
#     data_name="validation_0" # Refers to the first item in eval_set
# )

xgb_es.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)]
    # Removed callbacks as it caused a TypeError
)
preds_es = xgb_es.predict(X_test)

# If no early stopping, best_iteration is typically n_estimators - 1
# Access xgb_es.n_estimators since best_iteration is not available without early stopping
print(f"Number of estimators: {xgb_es.n_estimators}")
print(f"Accuracy: {accuracy_score(y_test, preds_es):.1%}")

[0]	validation_0-logloss:0.47321
[1]	validation_0-logloss:0.45288
[2]	validation_0-logloss:0.42852
[3]	validation_0-logloss:0.41302
[4]	validation_0-logloss:0.39269
[5]	validation_0-logloss:0.37405
[6]	validation_0-logloss:0.35797
[7]	validation_0-logloss:0.35790
[8]	validation_0-logloss:0.35521
[9]	validation_0-logloss:0.34085
[10]	validation_0-logloss:0.33806
[11]	validation_0-logloss:0.32476
[12]	validation_0-logloss:0.32308
[13]	validation_0-logloss:0.31360
[14]	validation_0-logloss:0.30218
[15]	validation_0-logloss:0.30080
[16]	validation_0-logloss:0.29103
[17]	validation_0-logloss:0.28115
[18]	validation_0-logloss:0.27248
[19]	validation_0-logloss:0.26566
[20]	validation_0-logloss:0.25925
[21]	validation_0-logloss:0.25398
[22]	validation_0-logloss:0.25289
[23]	validation_0-logloss:0.24739
[24]	validation_0-logloss:0.24661
[25]	validation_0-logloss:0.24591
[26]	validation_0-logloss:0.24177
[27]	validation_0-logloss:0.23603
[28]	validation_0-logloss:0.23188
[29]	validation_0-loglos

In [14]:
xgb_es = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

# The early_stopping_callback definition remains but will not be used in the fit method.
# early_stopping_callback = xgb.callback.EarlyStopping(
#     rounds=50,
#     save_best=True,
#     maximize=False, # For logloss, we want to minimize it
#     data_name="validation_0" # Refers to the first item in eval_set
# )

xgb_es.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)]
    # Removed callbacks as it caused a TypeError
)

preds_es = xgb_es.predict(X_test)

# Access xgb_es.n_estimators since best_iteration is not available without early stopping
print(f"Number of estimators: {xgb_es.n_estimators}")
print(f"Accuracy: {accuracy_score(y_test, preds_es):.1%}")

[0]	validation_0-logloss:0.47321
[1]	validation_0-logloss:0.45288
[2]	validation_0-logloss:0.42852
[3]	validation_0-logloss:0.41302
[4]	validation_0-logloss:0.39269
[5]	validation_0-logloss:0.37405
[6]	validation_0-logloss:0.35797
[7]	validation_0-logloss:0.35790
[8]	validation_0-logloss:0.35521
[9]	validation_0-logloss:0.34085
[10]	validation_0-logloss:0.33806
[11]	validation_0-logloss:0.32476
[12]	validation_0-logloss:0.32308
[13]	validation_0-logloss:0.31360
[14]	validation_0-logloss:0.30218
[15]	validation_0-logloss:0.30080
[16]	validation_0-logloss:0.29103
[17]	validation_0-logloss:0.28115
[18]	validation_0-logloss:0.27248
[19]	validation_0-logloss:0.26566
[20]	validation_0-logloss:0.25925
[21]	validation_0-logloss:0.25398
[22]	validation_0-logloss:0.25289
[23]	validation_0-logloss:0.24739
[24]	validation_0-logloss:0.24661
[25]	validation_0-logloss:0.24591
[26]	validation_0-logloss:0.24177
[27]	validation_0-logloss:0.23603
[28]	validation_0-logloss:0.23188
[29]	validation_0-loglos

In [16]:
xgb_cv = XGBClassifier(
    n_estimator=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
cv_scores = cross_val_score(xgb_cv, X, y, cv=5, scoring='accuracy')
print("=== 5-Fold Cross-Validation Results ===")
print(f"Scores per fold: {cv_scores.round(3)}")
print(f"Mean CV Accuracy: {cv_scores.mean():.1%}")
print(f"Standard Deviation: {cv_scores.std():.1%}")
print(f"\n Reliable estimate: {cv_scores.mean():.1%}  {cv_scores.std():.1%}")

=== 5-Fold Cross-Validation Results ===
Scores per fold: [0.93  0.938 0.938 0.948 0.962]
Mean CV Accuracy: 94.3%
Standard Deviation: 1.1%

 Reliable estimate: 94.3%  1.1%
